### xgboost is not available in sklearn directly so we need to download it first

In [3]:
!pip install xgboost

In [4]:
import xgboost as xgb
import pandas as pd
from sklearn.datasets import make_classification # used to generate a random sample for CLASSIFICATION problems 
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier

---
## <u>Generate Dataset</u>

In [5]:
X, y = make_classification(   # no encoding or scaling need (scaling only needed if we make a penalization model like L1/L2)
    n_samples = 10000,
    n_features = 15,
    n_informative = 12,
    n_redundant = 2,
    n_classes = 2,            # binary classification 
    random_state = 42
)

---
## <u>Train Test Split</u>

In [6]:
X = pd.DataFrame(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

X_train.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
9254,3.223624,0.861537,2.381547,1.395780,4.014657,-1.527077,3.048176,1.264915,3.135539,3.286052,1.067258,-5.254813,-1.072045,-1.154300,-2.089999
1561,-3.577916,0.271841,-2.039949,4.737522,2.042567,2.968547,3.149826,4.125000,2.379183,0.855071,6.071050,7.298252,-6.449665,0.570726,-1.825491
1670,-3.274038,-1.223501,-2.915686,1.914367,-1.074814,0.016904,-1.189167,-1.681084,-0.641283,1.205698,-4.650563,-1.960073,4.951817,1.183464,-2.243373
6087,0.850901,-0.702375,-0.947497,2.046136,2.498039,1.364085,3.353895,-2.033510,2.892577,2.547747,2.607806,0.766182,-1.234793,0.483933,-1.407348
6669,0.471267,-0.092617,-2.245596,-1.257869,2.061824,-2.211428,0.891296,1.083817,-1.848658,0.857356,-1.442283,-0.577073,1.419910,0.391048,1.179483


---
## <u>Create and Train model to Predict</u>

In [7]:
# read more about xgb_classifier at https://xgboost.readthedocs.io/en/stable/python/python_api.html

xgb_classifier = xgb.XGBClassifier( 
    n_estimators = 50,
    max_depth = 3,
    learning_rate = 0.1,
    eval_metric = "logloss", # Logloss evaluates the uncertainty of the predicted probabilities across the entire dataset
    random_state = 42
)

xgb_classifier.fit(X_train, y_train)

y_train_pred = xgb_classifier.predict(X_train)
y_test_pred = xgb_classifier.predict(X_test)

---
## <u>Evaluate</u>

In [8]:
print("For XGBOOST (baseline) :-\n")

print("\nTraining scores :-")
print("Train Accuracy : ", accuracy_score(y_train, y_train_pred))
print("Train precision : ", precision_score(y_train, y_train_pred))
print("Train recall : ", recall_score(y_train, y_train_pred))
print("Train F1  : ", f1_score(y_train, y_train_pred))
print("Train confusion matrix : \n", confusion_matrix(y_train, y_train_pred))


print("\nTesting scores :-")
print("Test Accuracy : ", accuracy_score(y_test, y_test_pred))
print("Test precision : ", precision_score(y_test, y_test_pred))
print("Test recall : ", recall_score(y_test, y_test_pred))
print("Test F1 : ", f1_score(y_test, y_test_pred))
print("Test confusion matrix : \n", confusion_matrix(y_test, y_test_pred))

For XGBOOST (baseline) :-


Training scores :-
Train Accuracy :  0.92075
Train precision :  0.9007759228779685
Train recall :  0.9475636903289636
Train F1  :  0.9235776277724205
Train confusion matrix : 
 [[3535  422]
 [ 212 3831]]

Testing scores :-
Test Accuracy :  0.895
Test precision :  0.8697813121272365
Test recall :  0.9171907756813418
Test F1 :  0.8928571428571429
Test confusion matrix : 
 [[915 131]
 [ 79 875]]


---
## <u>Hyperparamter Tuning</u>

In [11]:
# 1. make pipeline
steps = [("xgbc", xgb.XGBClassifier(random_state = 42))]
pipeline = Pipeline(steps)

# 2. make paramter gird
param_grid = {
    "xgbc__n_estimators" : [50,100,150],
    "xgbc__max_depth" : [2,4,6],
    "xgbc__learning_rate" : [0.1, 0.01, 0.001],
    "xgbc__subsample" : [0.5,0.7,0.9],       # train on x percent of data only 
    "xgbc__eval_metric" : ["logloss", "map"] # map : mean average precision
}

# 3. cross validation
xgb_classifier_cv = RandomizedSearchCV(
    pipeline,
    param_grid,
    n_jobs = -1,
    cv = 5,
    scoring = "accuracy",
    random_state = 42
)

# 4. create and fit the model
xgb_classifier_cv.fit(X_train, y_train)

# 5. make predictions
y_train_pred = xgb_classifier_cv.predict(X_train)
y_test_pred = xgb_classifier_cv.predict(X_test)

# 6. Evaluate
print("For XGBOOST (hyperparamter tuning) :-\n")

print("\nTraining scores :-")
print("Train Accuracy : ", accuracy_score(y_train, y_train_pred))
print("Train precision : ", precision_score(y_train, y_train_pred))
print("Train recall : ", recall_score(y_train, y_train_pred))
print("Train F1  : ", f1_score(y_train, y_train_pred))
print("Train confusion matrix : \n", confusion_matrix(y_train, y_train_pred))


print("\nTesting scores :-")
print("Test Accuracy : ", accuracy_score(y_test, y_test_pred))
print("Test precision : ", precision_score(y_test, y_test_pred))
print("Test recall : ", recall_score(y_test, y_test_pred))
print("Test F1 : ", f1_score(y_test, y_test_pred))
print("Test confusion matrix : \n", confusion_matrix(y_test, y_test_pred))

For XGBOOST (hyperparamter tuning) :-


Training scores :-
Train Accuracy :  0.976125
Train precision :  0.9683852140077821
Train recall :  0.9849121939154093
Train F1  :  0.9765787860208461
Train confusion matrix : 
 [[3827  130]
 [  61 3982]]

Testing scores :-
Test Accuracy :  0.946
Test precision :  0.9325153374233128
Test recall :  0.9559748427672956
Test F1 :  0.9440993788819876
Test confusion matrix : 
 [[980  66]
 [ 42 912]]
